In [1]:
import h5py


H5_PATHs = ["../outputs/results/simulation_results_INSIDE_fgp_tabu_global_benchmark_racetrack_4_[2]pzs_[8]perpz_NODAG.h5",  "../outputs/results/simulation_results_OUTSIDE_fgp_tabu_global_benchmark_final_[4]pzs_[4]perpz_NODAG.h5", "../outputs/results/simulation_results_INSIDE_fgp_tabu_global_benchmark_dense_4_[8]pzs_[2]perpz_NODAG.h5"]

for H5_PATH in H5_PATHs:
    with h5py.File(H5_PATH, "r") as f:
        results = f.get("results")
        if results is None:
            raise ValueError("No 'results' group found in H5 file")

        run_names = list(results.keys())
        print(f"Runs: {len(run_names)}")
        for run_name in run_names[:5]:
            attrs = dict(results[run_name].attrs)
            print(run_name, attrs)


Runs: 73
run_0000 {'algo_balance_penalty': np.float64(0.5), 'algo_candidate_list_length': 'None', 'algo_max_iterations_factor': np.int64(20), 'algo_seed': np.int64(0), 'algorithm_name': 'qft_nativegates_quantinuum_qiskit_opt2', 'cost_after': np.float64(432.0), 'cost_before': np.float64(744.0), 'cpu_time_seconds': np.float64(0.833603), 'enable_memory_zone_manager': np.False_, 'enforce_slice_plan': np.False_, 'final_timesteps': np.int64(395), 'gate_count_1q': np.int64(254), 'gate_count_2q': np.int64(105), 'grid_size': np.int64(4), 'ions_per_pz': np.int64(8), 'move_distance_total': np.float64(432.0), 'mz_trap_size': np.int64(4), 'num_ions': np.int64(10), 'num_pzs': np.int64(2), 'optimize_params': np.False_, 'partitioning_algorithm': 'fgp_tabu_global', 'plot': np.False_, 'save': np.False_, 'success': np.True_, 'timesteps_lower_bound': np.int64(284), 'use_dag': np.False_}
run_0001 {'algo_balance_penalty': np.float64(0.5), 'algo_candidate_list_length': 'None', 'algo_max_iterations_factor': n

In [2]:
import csv
import h5py



CSV_PATH = "../outputs/results/simulation_results_fgp_tabu_global_benchmark_combined.csv"

columns = [
    "algorithm_name",
    "partitioning_algorithm",
    "gate_count_1q",
    "gate_count_2q",
    "num_ions",
    "gate_count",
    "success",
    "final_timesteps",
    "cpu_time_seconds",
    "idle_count",
    "gate_time_one_qubit",
    "gate_time_two_qubit",
    "num_pzs",
    "max_ions_per_pz",
    "arch",
]

with open(CSV_PATH, "w", newline="", encoding="utf-8") as csvfile:
    
    writer = csv.DictWriter(csvfile, fieldnames=columns)
    writer.writeheader()

    for H5_PATH in H5_PATHs:
    

        with h5py.File(H5_PATH, "r") as f:
            results = f.get("results")
            if results is None:
                raise ValueError("No 'results' group found in H5 file")

        
            for run_name in results.keys():
                attrs = results[run_name].attrs
                row = {key: attrs.get(key) for key in columns}
                writer.writerow(row)

print(f"Wrote {CSV_PATH}")


Wrote ../outputs/results/simulation_results_fgp_tabu_global_benchmark_combined.csv


In [ ]:
import csv
from collections import defaultdict


CSV_PATH = "../outputs/results/simulation_results_fgp_tabu_global_benchmark_combined.csv"
LATEX_TXT_PATH = "../outputs/results/fgp_latex_table.txt"

BENCHMARK_LABELS = {
    "qft": "QFT",
    "qpeexact": "QPEexact",
    "qaoa": "QAOA",
    "random": "Random",
}
BENCHMARK_ORDER = ["QFT", "QPEexact", "QAOA", "Random"]


def normalize_benchmark(name: str) -> str:
    base = name.split("nativegates")[0].rstrip("_")
    return BENCHMARK_LABELS.get(base, base)


def to_int(value):
    if value in (None, ""):
        return None
    return int(value)


def to_float(value):
    if value in (None, ""):
        return None
    return float(value)


# ----------------------------
# CSV aggregation
# ----------------------------

rows = {}

with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        architecture = (row.get("num_pzs") or "").strip()
        benchmark = normalize_benchmark(row.get("algorithm_name", ""))
        qubits = to_int(row.get("num_ions"))

        if architecture == "" or benchmark == "" or qubits is None:
            continue

        key = (architecture, benchmark, qubits)

        entry = rows.setdefault(
            key,
            {
                "architecture": architecture,
                "benchmark": benchmark,
                "qubits": qubits,
                "g1": None,
                "g2": None,
                "t_static": None,
                "t_fgp": None,
                "cpu_static": None,
                "cpu_fgp": None,
            },
        )

        g1 = to_int(row.get("gate_count_1q"))
        g2 = to_int(row.get("gate_count_2q"))
        if entry["g1"] is None and g1 is not None:
            entry["g1"] = g1
        if entry["g2"] is None and g2 is not None:
            entry["g2"] = g2

        timesteps = to_int(row.get("final_timesteps"))
        cpu = to_float(row.get("cpu_time_seconds"))

        part = (row.get("partitioning_algorithm") or "").strip().lower()
        if part == "none":
            entry["t_static"] = timesteps
            entry["cpu_static"] = cpu
        elif part == "fgp_tabu_global":
            entry["t_fgp"] = timesteps
            entry["cpu_fgp"] = cpu


# ----------------------------
# Derived quantities
# ----------------------------

for entry in rows.values():
    if entry["t_fgp"] is not None and entry["t_static"] not in (None, 0):
        entry["pct_timesteps"] = (
            100.0 * (entry["t_fgp"] - entry["t_static"]) / entry["t_static"]
        )
    else:
        entry["pct_timesteps"] = None

    if entry["cpu_fgp"] is not None and entry["cpu_static"] not in (None, 0):
        entry["pct_cpu"] = (
            100.0 * (entry["cpu_fgp"] - entry["cpu_static"]) / entry["cpu_static"]
        )
    else:
        entry["pct_cpu"] = None


# ----------------------------
# Formatting helpers
# ----------------------------

def fmt_int(value):
    return "-" if value is None else f"{int(value)}"


def fmt_float(value):
    return "-" if value is None else f"{value:.2f}"


def fmt_fgp_with_delta(t_fgp, t_static):
    if t_fgp is None:
        return "-"
    if t_static in (None, 0):
        return f"{int(t_fgp)}"

    delta = 100.0 * (t_fgp - t_static) / t_static
    sign = "+" if delta > 0 else ""
    perc = rf"{sign}{delta:.1f}\%"

    if delta < 0:
        perc = rf"\textcolor{{Green}}{{\textbf{{{perc}}}}}"
    elif delta > 0:
        perc = rf"\textcolor{{Red}}{{{perc}}}"

    return f"{int(t_fgp)} ({perc})"


def fmt_fgp_cpu_with_delta(cpu_fgp, cpu_static):
    if cpu_fgp is None:
        return "-"
    if cpu_static in (None, 0):
        return f"{cpu_fgp:.2f}"

    delta = 100.0 * (cpu_fgp - cpu_static) / cpu_static
    sign = "+" if delta > 0 else ""
    perc = rf"{sign}{delta:.1f}\%"

    if delta < 0:
        perc = rf"\textcolor{{Green}}{{\textbf{{{perc}}}}}"
    elif delta > 0:
        perc = rf"\textcolor{{Red}}{{{perc}}}"

    return f"{cpu_fgp:.2f} ({perc})"


def fmt_avg_pct(pct):
    if pct is None:
        return "-"
    sign = "+" if pct > 0 else ""
    txt = rf"{sign}{pct:.1f}\%"

    if pct < 0:
        txt = rf"\textcolor{{Green}}{{\textbf{{{txt}}}}}"
    elif pct > 0:
        txt = rf"\textcolor{{Red}}{{{txt}}}"

    return txt


# ----------------------------
# Sorting
# ----------------------------

def sort_key(item):
    architecture, benchmark, qubits = item[0]
    bench_order = (
        BENCHMARK_ORDER.index(benchmark)
        if benchmark in BENCHMARK_ORDER
        else len(BENCHMARK_ORDER)
    )
    return (architecture, bench_order, benchmark, qubits)


sorted_items = sorted(rows.items(), key=sort_key)


# ----------------------------
# Architecture averages
# ----------------------------

arch_stats = defaultdict(lambda: {"pct_timesteps": [], "pct_cpu": []})

for entry in rows.values():
    arch = entry["architecture"]
    if entry["pct_timesteps"] is not None:
        arch_stats[arch]["pct_timesteps"].append(entry["pct_timesteps"])
    if entry["pct_cpu"] is not None:
        arch_stats[arch]["pct_cpu"].append(entry["pct_cpu"])

arch_avg = {
    arch: {
        "pct_timesteps": (
            sum(v["pct_timesteps"]) / len(v["pct_timesteps"])
            if v["pct_timesteps"] else None
        ),
        "pct_cpu": (
            sum(v["pct_cpu"]) / len(v["pct_cpu"])
            if v["pct_cpu"] else None
        ),
    }
    for arch, v in arch_stats.items()
}


# ----------------------------
# LaTeX generation
# ----------------------------

lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Comparison of static and fine-grained partitioning across benchmarks.}",
    r"\label{tab:benchmark_results}",
    r"\begin{tabular}{lllrr|rr|rr}",
    r"\hline",
    r"Architecture & Benchmark & Qubits & \#1Q Gates & \#2Q Gates & "
    r"\multicolumn{2}{c|}{Static} & \multicolumn{2}{c}{FGP} \\",
    r" & & & & & Timesteps & Comp time [$s$] & Timesteps & Comp time [$s$] \\",
    r"\hline",
]

last_arch = None
last_benchmark = None

for _, entry in sorted_items:
    if last_arch is not None and entry["architecture"] != last_arch:
        avg = arch_avg[last_arch]
        lines.append(
            rf"\hline"
            rf"\textbf{{$\varnothing$}} & "
            rf"\multicolumn{{4}}{{r}}{{}} & "
            rf"\multicolumn{{2}}{{c|}}{{}} & "
            f"{fmt_avg_pct(avg['pct_timesteps'])} & "
            rf"{fmt_avg_pct(avg['pct_cpu'])} \\"
        )
        lines.append(r"\hline\hline")
    elif last_benchmark is not None and entry["benchmark"] != last_benchmark:
        lines.append(r"\hline")

    lines.append(
        (
            "{arch} & {benchmark} & {qubits} & {g1} & {g2} & "
            r"{t_static} & {cpu_static} & {t_fgp} & {cpu_fgp} \\"
        ).format(
            arch=entry["architecture"],
            benchmark=entry["benchmark"],
            qubits=fmt_int(entry["qubits"]),
            g1=fmt_int(entry["g1"]),
            g2=fmt_int(entry["g2"]),
            t_static=fmt_int(entry["t_static"]),
            cpu_static=fmt_float(entry["cpu_static"]),
            t_fgp=fmt_fgp_with_delta(entry["t_fgp"], entry["t_static"]),
            cpu_fgp=fmt_fgp_cpu_with_delta(entry["cpu_fgp"], entry["cpu_static"]),
        )
    )

    last_arch = entry["architecture"]
    last_benchmark = entry["benchmark"]

if last_arch is not None:
    avg = arch_avg[last_arch]
    lines.append(
        rf"\hline"
        rf"\textbf{{$\varnothing$}} & "
        rf"\multicolumn{{4}}{{r}}{{}} & "
        rf"\multicolumn{{2}}{{c|}}{{}} & "
        f"{fmt_avg_pct(avg['pct_timesteps'])} & "
        rf"{fmt_avg_pct(avg['pct_cpu'])} \\"
    )
    lines.append(r"\hline")

lines.extend([
    r"\end{tabular}",
    r"\end{table*}",
])

latex = "\n".join(lines)

print(latex)

with open(LATEX_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(latex)


\begin{table*}[t]
\centering
\caption{Comparison of static and fine-grained partitioning across benchmarks.}
\label{tab:benchmark_results}
\begin{tabular}{lllrr|rr|rr}
\hline
Architecture & Benchmark & Qubits & \#1Q Gates & \#2Q Gates & \multicolumn{2}{c|}{Static} & \multicolumn{2}{c}{FGP} \\
 & & & & & Timesteps & Comp time [$s$] & Timesteps & Comp time [$s$] \\
\hline
2 & QFT & 10 & 254 & 105 & 747 & 9.71 & 485 (\textcolor{Green}{\textbf{-35.1\%}}) & 1.42 (\textcolor{Green}{\textbf{-85.3\%}}) \\
2 & QFT & 30 & 1923 & 915 & 5545 & 559.47 & 3700 (\textcolor{Green}{\textbf{-33.3\%}}) & 35.81 (\textcolor{Green}{\textbf{-93.6\%}}) \\
2 & QFT & 60 & 5229 & 3630 & - & - & 12157 & 302.48 \\
\hline
2 & QPEexact & 10 & 214 & 98 & 584 & 7.33 & 366 (\textcolor{Green}{\textbf{-37.3\%}}) & 0.75 (\textcolor{Green}{\textbf{-89.8\%}}) \\
2 & QPEexact & 30 & 1826 & 910 & 7078 & 746.64 & 4637 (\textcolor{Green}{\textbf{-34.5\%}}) & 45.50 (\textcolor{Green}{\textbf{-93.9\%}}) \\
2 & QPEexact & 60 & 5218

In [35]:
import csv
import h5py
import numpy as np
from collections import defaultdict
from collections import Counter


CSV_PATH = "../outputs/results/simulation_results_fgp_tabu_global_benchmark_combined.csv"
LATEX_TXT_PATH = "../outputs/results/fgp_latex_table_new.txt"


# ----------------------------
# Export H5 → CSV
# ----------------------------

columns = [
    "algorithm_name",
    "partitioning_algorithm",
    "gate_count_1q",
    "gate_count_2q",
    "num_ions",
    "gate_count",
    "success",
    "final_timesteps",
    "cpu_time_seconds",
    "idle_count",
    "gate_time_one_qubit",
    "gate_time_two_qubit",
    "num_pzs",
    "max_ions_per_pz",
    "arch",
]

with open(CSV_PATH, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.DictWriter(csvfile, fieldnames=columns)
    writer.writeheader()

    for H5_PATH in H5_PATHs:

        with h5py.File(H5_PATH, "r") as f:
            results = f.get("results")

            for run_name in results.keys():
                attrs = results[run_name].attrs
                row = {key: attrs.get(key) for key in columns}
                writer.writerow(row)

print(f"Wrote {CSV_PATH}")


# ----------------------------
# Benchmark labels
# ----------------------------

BENCHMARK_LABELS = {
    "qft": "QFT",
    "qpeexact": "QPEexact",
    "qaoa": "QAOA",
    "random": "Random",
}

BENCHMARK_ORDER = ["QFT", "QPEexact", "QAOA", "Random"]


def normalize_benchmark(name: str) -> str:
    base = name.split("nativegates")[0].rstrip("_")
    return BENCHMARK_LABELS.get(base, base)


def to_int(v):
    return None if v in (None, "") else int(v)


def to_float(v):
    return None if v in (None, "") else float(v)


# ----------------------------
# Aggregation
# ----------------------------

rows = {}

with open(CSV_PATH, newline="", encoding="utf-8") as f:

    reader = csv.DictReader(f)

    for row in reader:

        num_pzs = (row.get("num_pzs") or "").strip()
        cap = (row.get("num_pzs") or "").strip()
        bench = normalize_benchmark(row.get("algorithm_name", ""))
        qubits = to_int(row.get("num_ions"))

        if arch == "" or bench == "" or qubits is None:
            continue
        else:
            arch = f"$k$ = {num_pzs}, $c$ = {cap}"

        key = (arch, bench, qubits)

        entry = rows.setdefault(
            key,
            {
                "architecture": arch,
                "benchmark": bench,
                "qubits": qubits,
                "t_static": [],
                "t_fgp": [],
                "cpu_static": [],
                "cpu_fgp": [],
            },
        )

        timesteps = to_int(row.get("final_timesteps"))
        cpu = to_float(row.get("cpu_time_seconds"))

        part = (row.get("partitioning_algorithm") or "").strip().lower()

        if part == "none":
            if timesteps is not None:
                entry["t_static"].append(timesteps)
            if cpu is not None:
                entry["cpu_static"].append(cpu)

        elif part == "fgp_tabu_global":
            if timesteps is not None:
                entry["t_fgp"].append(timesteps)
            if cpu is not None:
                entry["cpu_fgp"].append(cpu)


# ----------------------------
# Statistics
# ----------------------------

def mean(x):
    return float(np.mean(x)) if len(x) > 0 else None


def std(x):
    return float(np.std(x)) if len(x) > 1 else None


for e in rows.values():

    e["t_static_mean"] = mean(e["t_static"])
    e["t_fgp_mean"] = mean(e["t_fgp"])

    e["cpu_static_mean"] = mean(e["cpu_static"])
    e["cpu_fgp_mean"] = mean(e["cpu_fgp"])

    if e["t_static_mean"] not in (None, 0) and e["t_fgp_mean"] is not None:
        e["pct_timesteps"] = 100 * (e["t_fgp_mean"] - e["t_static_mean"]) / e["t_static_mean"]
    else:
        e["pct_timesteps"] = None

    if e["cpu_static_mean"] not in (None, 0) and e["cpu_fgp_mean"] is not None:
        e["pct_cpu"] = 100 * (e["cpu_fgp_mean"] - e["cpu_static_mean"]) / e["cpu_static_mean"]
    else:
        e["pct_cpu"] = None


# ----------------------------
# Formatting helpers
# ----------------------------

def fmt_int(x):
    return "-" if x is None else f"{int(round(x))}"


def fmt_float(x):
    return "-" if x is None else f"{x:.2f}"


def fmt_mean_std(values, fmt_func, show_std=False):

    if len(values) == 0:
        return "-"

    m = mean(values)
    s = std(values)

    if show_std and s is not None:
        return f"${fmt_func(m)} \\pm {fmt_func(s)}$"

    return fmt_func(m)


def fmt_pct(x):

    if x is None:
        return "-"

    color = "ForestGreen" if x < 0 else "BrickRed"
    return f"\\textcolor{{{color}}}{{{x:.1f}\\%}}"


# ----------------------------
# Sorting
# ----------------------------

def sort_key(item):

    arch, bench, qubits = item[0]

    bench_order = (
        BENCHMARK_ORDER.index(bench)
        if bench in BENCHMARK_ORDER
        else len(BENCHMARK_ORDER)
    )

    return (arch, bench_order, bench, qubits)


sorted_items = sorted(rows.items(), key=sort_key)


arch_counts = Counter()
bench_counts = Counter()

for (arch, bench, _), _ in sorted_items:
    arch_counts[arch] += 1
    bench_counts[(arch, bench)] += 1

# ----------------------------
# Architecture averages
# ----------------------------

arch_stats = defaultdict(lambda: {"t": [], "cpu": []})

for e in rows.values():

    arch = e["architecture"]

    if e["pct_timesteps"] is not None:
        arch_stats[arch]["t"].append(e["pct_timesteps"])

    if e["pct_cpu"] is not None:
        arch_stats[arch]["cpu"].append(e["pct_cpu"])


arch_avg = {
    arch: {
        "pct_timesteps": mean(v["t"]),
        "pct_cpu": mean(v["cpu"]),
    }
    for arch, v in arch_stats.items()
}

# ----------------------------
# LaTeX table
# ----------------------------

from collections import Counter

# count rows per architecture and benchmark
arch_counts = Counter()
bench_counts = Counter()

for (arch, bench, _), _ in sorted_items:
    arch_counts[arch] += 1
    bench_counts[(arch, bench)] += 1


lines = [
r"\begin{table*}[t]",
r"\centering",
r"\caption{Compilation performance comparing static and fine-grained partitioning (FGP).}",
r"\label{tab:benchmark_results}",
r"\begin{tabular}{c|ll|rrr|rrr}",
r"\hline \hline",
r"Architecture & Benchmark & Qubits & \multicolumn{3}{c|}{Timesteps} & \multicolumn{3}{c}{Compile time [$s$]} \\",
r" & & & Static & FGP & $\Delta$ & Static & FGP & $\Delta$ \\",
r"\hline",
]

last_arch = None
last_bench = None

arch_seen = defaultdict(int)
bench_seen = defaultdict(int)

for (arch, bench, qubits), e in sorted_items:

    # architecture change
    if last_arch and arch != last_arch:

        avg = arch_avg[last_arch]

        lines.append(r"\hline")

    # benchmark change
    elif last_bench and (arch, bench) != (last_arch, last_bench):
        lines.append(r"\cline{2-9}")

    # architecture cell
    if arch_seen[arch] == 0:
        k = int(arch.split(",")[0].split("=")[1].strip())
        arch_cell = (
            rf"\multirow{{{arch_counts[arch]}}}{{*}}"
            rf"{{{{\includegraphics[scale = 0.4]{{figures/{k}pz_arch.pdf}}}}}}")
    else:
        arch_cell = ""

    # benchmark cell
    if bench_seen[(arch, bench)] == 0:
        bench_cell = rf"\multirow{{{bench_counts[(arch, bench)]}}}{{*}}{{{bench}}}"
    else:
        bench_cell = ""

    arch_seen[arch] += 1
    bench_seen[(arch, bench)] += 1

    lines.append(
        (
            "{arch} & {bench} & {q} & "
            "{t_s} & {t_f} & {dt} & "
            "{c_s} & {c_f} & {dc} \\\\"
        ).format(
            arch=arch_cell,
            bench=bench_cell,
            q=fmt_int(e["qubits"]),

            t_s=fmt_mean_std(e["t_static"], fmt_int),
            t_f=fmt_mean_std(e["t_fgp"], fmt_int, True),
            dt=fmt_pct(e["pct_timesteps"]),

            c_s=fmt_mean_std(e["cpu_static"], fmt_float),
            c_f=fmt_mean_std(e["cpu_fgp"], fmt_float, True),
            dc=fmt_pct(e["pct_cpu"]),
        )
    )

    last_arch = arch
    last_bench = bench


if last_arch:

    avg = arch_avg[last_arch]

    lines.append(
        rf"\hline \hline"
        rf"\textbf{{$\varnothing$}} & "
        rf"\multicolumn{{2}}{{r|}}{{}} & "
        rf"\multicolumn{{2}}{{c}}{{}} & "
        f"{fmt_pct(avg['pct_timesteps'])} & "
        rf"\multicolumn{{2}}{{c}}{{}} & "
        rf"{fmt_pct(avg['pct_cpu'])} \\"
    )

    lines.append(r"\hline")


lines += [
r"\end{tabular}",
r"\end{table*}",
]

latex = "\n".join(lines)

print(latex)

with open(LATEX_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(latex)

Wrote ../outputs/results/simulation_results_fgp_tabu_global_benchmark_combined.csv
\begin{table*}[t]
\centering
\caption{Compilation performance comparing static and fine-grained partitioning (FGP).}
\label{tab:benchmark_results}
\begin{tabular}{c|ll|rrr|rrr}
\hline \hline
Architecture & Benchmark & Qubits & \multicolumn{3}{c|}{Timesteps} & \multicolumn{3}{c}{Compile time [$s$]} \\
 & & & Static & FGP & $\Delta$ & Static & FGP & $\Delta$ \\
\hline
\multirow{12}{*}{{\includegraphics[scale = 0.4]{figures/2pz_arch.pdf}}} & \multirow{3}{*}{QFT} & 10 & 747 & $400 \pm 36$ & \textcolor{ForestGreen}{-46.4\%} & 9.71 & $1.00 \pm 0.22$ & \textcolor{ForestGreen}{-89.7\%} \\
 &  & 30 & 5545 & $3773 \pm 217$ & \textcolor{ForestGreen}{-32.0\%} & 559.47 & $35.80 \pm 1.65$ & \textcolor{ForestGreen}{-93.6\%} \\
 &  & 60 & - & $12343 \pm 202$ & - & - & $304.60 \pm 3.74$ & - \\
\cline{2-9}
 & \multirow{3}{*}{QPEexact} & 10 & 584 & $354 \pm 13$ & \textcolor{ForestGreen}{-39.4\%} & 7.33 & $0.73 \pm 0.03$ & 